# Project 1 — Data Cleaning & Preparation

## Project Objective

The objective of this project is to clean and prepare a raw e-commerce
dataset for reliable downstream analysis.

The data-cleaning process focuses on:

- Identifying missing values
- Detecting and removing duplicate records
- Checking for duplicate OrderIDs
- Correcting data types and formats
- Validating dates, numerical fields, and text fields
- Performing post-cleaning quality checks
- Exporting the final cleaned dataset

The raw dataset is preserved separately so that all transformations
remain reproducible and auditable.

## 1. Load Raw Dataset

The raw Excel dataset is loaded into a pandas DataFrame without modifying
the original source file.

In [4]:
# ============================================================
# 1. LOAD RAW DATASET
# ============================================================

import pandas as pd
import numpy as np

# Install the Excel reader if it is not available
!pip install openpyxl -q

file_path = "Dataset for Data Analytics.xlsx"

df_raw = pd.read_excel(file_path, engine="openpyxl")

print("Dataset loaded successfully.")
print(f"Rows    : {df_raw.shape[0]:,}")
print(f"Columns : {df_raw.shape[1]:,}")

You should consider upgrading via the '/opt/conda/bin/python3 -m pip install --upgrade pip' command.
Dataset loaded successfully.
Rows    : 1,200
Columns : 14


In [5]:
# Display the first five records

display(df_raw.head())

,OrderID,Date,CustomerID,Product,Quantity,UnitPrice,ShippingAddress,PaymentMethod,OrderStatus,TrackingNumber,ItemsInCart,CouponCode,ReferralSource,TotalPrice
0,ORD200000,2023-01-04,C72649,Monitor,5,570.62,928 Main St,Debit Card,Shipped,TRK37947903,7,SAVE10,Instagram,2853.10
1,ORD200001,2024-08-23,C75739,Phone,2,151.35,823 Main St,Online,Shipped,TRK91186779,3,SAVE10,Referral,302.70
2,ORD200002,2024-02-27,C81728,Tablet,5,550.68,512 Main St,Credit Card,Cancelled,TRK42903982,8,FREESHIP,Email,2753.40
3,ORD200003,2023-10-15,C33540,Chair,1,273.19,275 Main St,Debit Card,Returned,TRK62788070,5,SAVE10,Facebook,273.19
4,ORD200004,2025-05-08,C81840,Printer,4,626.01,668 Main St,Online,Delivered,TRK29241424,8,SAVE10,Email,2504.04


In [6]:
# Display column names

print("Column names:")
print(df_raw.columns.tolist())

Column names:
['OrderID', 'Date', 'CustomerID', 'Product', 'Quantity', 'UnitPrice', 'ShippingAddress', 'PaymentMethod', 'OrderStatus', 'TrackingNumber', 'ItemsInCart', 'CouponCode', 'ReferralSource', 'TotalPrice']


## 2. Initial Data Inspection

Before applying any cleaning operations, the dataset is inspected to
understand its structure, data types, missing values, and duplicate records.

In [7]:
# ============================================================
# 2. INITIAL DATA INSPECTION
# ============================================================

print("Dataset shape:")
print(df_raw.shape)

print("\nData types:")
display(df_raw.dtypes.to_frame("Data Type"))

print("\nMissing values:")
display(df_raw.isnull().sum().to_frame("Missing Values"))

print("\nDuplicate rows:")
print(df_raw.duplicated().sum())

Dataset shape:
(1200, 14)

Data types:


,Data Type
OrderID,object
Date,datetime64[ns]
CustomerID,object
Product,object
Quantity,int64
UnitPrice,float64
ShippingAddress,object
PaymentMethod,object
OrderStatus,object
TrackingNumber,object



Missing values:


,Missing Values
OrderID,0
Date,0
CustomerID,0
Product,0
Quantity,0
UnitPrice,0
ShippingAddress,0
PaymentMethod,0
OrderStatus,0
TrackingNumber,0



Duplicate rows:
0


## 3. Missing Value Analysis

Missing values are assessed column by column to determine whether they
represent data-quality problems or valid absence of information.

The `CouponCode` field contains missing values. Because a customer may
legitimately place an order without using a coupon, these missing values
are treated as "No Coupon" rather than removing the corresponding records.

In [8]:
# ============================================================
# 3. MISSING VALUE ANALYSIS
# ============================================================

missing_summary = pd.DataFrame({
    "Missing Count": df_raw.isnull().sum(),
    "Missing Percentage": (
        df_raw.isnull().mean() * 100
    ).round(2)
})

missing_summary = missing_summary[
    missing_summary["Missing Count"] > 0
].sort_values(
    "Missing Count",
    ascending=False
)

display(missing_summary)

,Missing Count,Missing Percentage
CouponCode,309,25.75


### Missing Value Treatment

The missing `CouponCode` values are retained because the absence of a
coupon is a valid business condition rather than evidence of a missing
transaction.

The values will therefore be replaced with the explicit category
`No Coupon`, preserving all 1,200 observations.

In [10]:
# ============================================================
# 3.1 HANDLE MISSING COUPON CODES
# ============================================================

df_clean = df_raw.copy()

df_clean["CouponCode"] = df_clean["CouponCode"].fillna("No Coupon")

print("Missing CouponCode values after treatment:",
      df_clean["CouponCode"].isnull().sum())

Missing CouponCode values after treatment: 0


In [11]:
# Verify row count was preserved

print("Original rows :", len(df_raw))
print("Cleaned rows  :", len(df_clean))

assert len(df_raw) == len(df_clean)

print("\nAll original observations were preserved.")

Original rows : 1200
Cleaned rows  : 1200

All original observations were preserved.


## 4. Duplicate Analysis

Duplicate records are evaluated at two levels:

1. Duplicate complete rows
2. Duplicate `OrderID` values

A unique `OrderID` is important because it represents the individual
transaction identifier.

In [12]:
# ============================================================
# 4. DUPLICATE ANALYSIS
# ============================================================

# Check duplicate complete rows
duplicate_rows = df_clean.duplicated().sum()

# Check duplicate OrderIDs
duplicate_order_ids = df_clean["OrderID"].duplicated().sum()

print(f"Duplicate complete rows : {duplicate_rows}")
print(f"Duplicate OrderIDs      : {duplicate_order_ids}")

Duplicate complete rows : 0
Duplicate OrderIDs      : 0


In [13]:
# Display duplicated OrderID records, if any

duplicate_id_records = df_clean[
    df_clean["OrderID"].duplicated(keep=False)
].sort_values("OrderID")

if len(duplicate_id_records) > 0:
    display(duplicate_id_records)
else:
    print("No duplicate OrderIDs found.")

No duplicate OrderIDs found.


### Duplicate Treatment

Duplicate complete rows are removed if identified.

Duplicate `OrderID` records are investigated separately because an
OrderID should uniquely identify an order. Any confirmed duplicate
transaction records would be reviewed before removal to avoid deleting
legitimate business records.

In [14]:
# ============================================================
# 4.1 REMOVE DUPLICATE COMPLETE ROWS
# ============================================================

before_duplicates = len(df_clean)

df_clean = df_clean.drop_duplicates().copy()

after_duplicates = len(df_clean)

print("Rows before duplicate removal :", before_duplicates)
print("Rows after duplicate removal  :", after_duplicates)
print("Duplicate rows removed        :", before_duplicates - after_duplicates)

Rows before duplicate removal : 1200
Rows after duplicate removal  : 1200
Duplicate rows removed        : 0


In [15]:
# Final duplicate validation

remaining_duplicate_rows = df_clean.duplicated().sum()
remaining_duplicate_ids = df_clean["OrderID"].duplicated().sum()

print("Final duplicate validation")
print("--------------------------")
print(f"Duplicate complete rows : {remaining_duplicate_rows}")
print(f"Duplicate OrderIDs      : {remaining_duplicate_ids}")

assert remaining_duplicate_rows == 0
assert remaining_duplicate_ids == 0

print("\nValidation passed: zero duplicate rows and zero duplicate OrderIDs.")

Final duplicate validation
--------------------------
Duplicate complete rows : 0
Duplicate OrderIDs      : 0

Validation passed: zero duplicate rows and zero duplicate OrderIDs.


## 5. Data Type & Format Validation

The cleaned dataset is checked for appropriate data types across date,
numeric, and text fields.

Particular attention is given to the `Date` column because correctly
formatted dates are an explicit project requirement.

In [17]:
# ============================================================
# 5. DATA TYPE & FORMAT VALIDATION
# ============================================================

print("Current data types:")
display(df_clean.dtypes.to_frame("Data Type"))

Current data types:


,Data Type
OrderID,object
Date,datetime64[ns]
CustomerID,object
Product,object
Quantity,int64
UnitPrice,float64
ShippingAddress,object
PaymentMethod,object
OrderStatus,object
TrackingNumber,object


In [18]:
# ============================================================
# 5.1 DATE VALIDATION
# ============================================================

print("Date column data type:")
print(df_clean["Date"].dtype)

print("\nMissing dates:")
print(df_clean["Date"].isna().sum())

print("\nInvalid dates:")
invalid_dates = pd.to_datetime(
    df_clean["Date"],
    errors="coerce"
).isna().sum()

print(invalid_dates)

Date column data type:
datetime64[ns]

Missing dates:
0

Invalid dates:
0


In [19]:
# ============================================================
# 5.2 NUMERIC COLUMN VALIDATION
# ============================================================

numeric_columns = [
    "Quantity",
    "UnitPrice",
    "ItemsInCart",
    "TotalPrice"
]

numeric_validation = pd.DataFrame({
    "Data Type": df_clean[numeric_columns].dtypes.astype(str),
    "Missing Values": df_clean[numeric_columns].isna().sum(),
    "Non-Numeric Values": [
        pd.to_numeric(df_clean[col], errors="coerce").isna().sum()
        for col in numeric_columns
    ]
})

display(numeric_validation)

,Data Type,Missing Values,Non-Numeric Values
Quantity,int64,0,0
UnitPrice,float64,0,0
ItemsInCart,int64,0,0
TotalPrice,float64,0,0


In [20]:
# ============================================================
# 5.3 TEXT COLUMN VALIDATION
# ============================================================

text_columns = [
    "OrderID",
    "CustomerID",
    "Product",
    "ShippingAddress",
    "PaymentMethod",
    "OrderStatus",
    "TrackingNumber",
    "CouponCode",
    "ReferralSource"
]

text_validation = pd.DataFrame({
    "Data Type": df_clean[text_columns].dtypes.astype(str),
    "Missing Values": df_clean[text_columns].isna().sum()
})

display(text_validation)

,Data Type,Missing Values
OrderID,object,0
CustomerID,object,0
Product,object,0
ShippingAddress,object,0
PaymentMethod,object,0
OrderStatus,object,0
TrackingNumber,object,0
CouponCode,object,0
ReferralSource,object,0


### Validation Result

The dataset is assessed for inappropriate data types and invalid values
before applying transformations.

The `Date` column is expected to use a datetime data type, while
transaction quantities, prices, and cart counts are expected to be
numeric. Identifier and categorical fields are treated as text.

## 6. Data Cleaning & Standardization

The dataset is cleaned and standardized to improve consistency and reliability for downstream analysis.

The cleaning process includes:
- Standardizing text fields by removing unnecessary leading and trailing whitespace.
- Standardizing categorical values to ensure consistent representation.
- Treating missing `CouponCode` values as the explicit category `No Coupon` because the absence of a coupon represents a valid business condition.
- Ensuring numeric and date fields retain their validated data types.
- Rechecking the dataset after cleaning to confirm that the transformations did not introduce new missing values or duplicate records.

In [21]:
# ============================================================
# 6. DATA CLEANING & STANDARDIZATION
# ============================================================

# Create a copy so the original cleaned dataset remains unchanged
df_clean = df_clean.copy()

# ------------------------------------------------------------
# 6.1 STANDARDIZE TEXT FIELDS
# ------------------------------------------------------------

text_columns = [
    "OrderID",
    "CustomerID",
    "Product",
    "ShippingAddress",
    "PaymentMethod",
    "OrderStatus",
    "TrackingNumber",
    "CouponCode",
    "ReferralSource"
]

for col in text_columns:
    df_clean[col] = df_clean[col].apply(
        lambda x: x.strip() if isinstance(x, str) else x
    )

print("Text fields standardized successfully.")

Text fields standardized successfully.


In [22]:
# ============================================================
# 6.2 STANDARDIZE CATEGORICAL VALUES
# ============================================================

categorical_columns = [
    "PaymentMethod",
    "OrderStatus",
    "ReferralSource"
]

for col in categorical_columns:
    df_clean[col] = df_clean[col].apply(
        lambda x: x.strip() if isinstance(x, str) else x
    )

print("Categorical fields standardized successfully.")

Categorical fields standardized successfully.


In [23]:
# ============================================================
# 6.3 VALIDATE NUMERIC AND DATE TYPES
# ============================================================

df_clean["Date"] = pd.to_datetime(
    df_clean["Date"],
    errors="coerce"
)

numeric_columns = [
    "Quantity",
    "UnitPrice",
    "ItemsInCart",
    "TotalPrice"
]

for col in numeric_columns:
    df_clean[col] = pd.to_numeric(
        df_clean[col],
        errors="coerce"
    )

print("Numeric and date fields validated successfully.")

Numeric and date fields validated successfully.


In [24]:
# ============================================================
# 6.4 POST-CLEANING QUALITY CHECK
# ============================================================

print("Dataset shape after cleaning:")
print(df_clean.shape)

print("\nMissing values after cleaning:")
display(
    df_clean.isna().sum().to_frame("Missing Values")
)

print("\nDuplicate rows after cleaning:")
print(df_clean.duplicated().sum())

Dataset shape after cleaning:
(1200, 14)

Missing values after cleaning:


,Missing Values
OrderID,0
Date,0
CustomerID,0
Product,0
Quantity,0
UnitPrice,0
ShippingAddress,0
PaymentMethod,0
OrderStatus,0
TrackingNumber,0



Duplicate rows after cleaning:
0


## 7. Data Consistency & Business Rule Validation

The cleaned dataset is evaluated against basic business rules to identify logically inconsistent transaction records.

The validation focuses on:
- Positive order quantities.
- Non-negative unit prices.
- Non-negative cart item counts.
- Non-negative total prices.
- Valid order and customer identifiers.
- Consistency between the recorded total price and the expected quantity × unit price relationship.

These checks help identify records that may be technically valid but logically inconsistent from a business perspective.

In [26]:
# ============================================================
# 7.1 NUMERIC BUSINESS RULE VALIDATION
# ============================================================

business_rule_checks = {
    "Quantity <= 0": (df_clean["Quantity"] <= 0).sum(),
    "UnitPrice < 0": (df_clean["UnitPrice"] < 0).sum(),
    "ItemsInCart < 0": (df_clean["ItemsInCart"] < 0).sum(),
    "TotalPrice < 0": (df_clean["TotalPrice"] < 0).sum()
}

business_rule_results = pd.DataFrame(
    business_rule_checks.items(),
    columns=["Validation Rule", "Violations"]
)

display(business_rule_results)

,Validation Rule,Violations
0,Quantity <= 0,0
1,UnitPrice < 0,0
2,ItemsInCart < 0,0
3,TotalPrice < 0,0


In [27]:
# ============================================================
# 7.2 IDENTIFIER VALIDATION
# ============================================================

identifier_checks = pd.DataFrame({
    "Column": ["OrderID", "CustomerID"],
    "Missing Values": [
        df_clean["OrderID"].isna().sum(),
        df_clean["CustomerID"].isna().sum()
    ],
    "Blank Values": [
        (df_clean["OrderID"].astype(str).str.strip() == "").sum(),
        (df_clean["CustomerID"].astype(str).str.strip() == "").sum()
    ]
})

display(identifier_checks)

,Column,Missing Values,Blank Values
0,OrderID,0,0
1,CustomerID,0,0


In [28]:
# ============================================================
# 7.3 TOTAL PRICE CONSISTENCY CHECK
# ============================================================

df_clean["CalculatedTotal"] = (
    df_clean["Quantity"] * df_clean["UnitPrice"]
)

df_clean["PriceDifference"] = (
    df_clean["TotalPrice"] - df_clean["CalculatedTotal"]
).abs()

price_mismatches = (
    df_clean["PriceDifference"] > 0.01
).sum()

print("Total price mismatches:", price_mismatches)

Total price mismatches: 0


In [29]:
# ============================================================
# 7.4 REMOVE TEMPORARY VALIDATION COLUMNS
# ============================================================

df_clean.drop(
    columns=["CalculatedTotal", "PriceDifference"],
    inplace=True
)

print("Temporary validation columns removed.")
print("Final column count:", len(df_clean.columns))

Temporary validation columns removed.
Final column count: 14


In [30]:
# ============================================================
# 7.5 SECTION 7 VALIDATION SUMMARY
# ============================================================

print("Dataset shape:", df_clean.shape)
print("Duplicate rows:", df_clean.duplicated().sum())
print("Total missing values:", df_clean.isna().sum().sum())

Dataset shape: (1200, 14)
Duplicate rows: 0
Total missing values: 0


## 8. Final Data Quality Check

The cleaned dataset is subjected to a final quality check to confirm that the required cleaning and validation steps have been completed successfully.

The final dataset is checked for:
- Dataset dimensions
- Duplicate records
- Missing values
- Correct data types

In [31]:
# ============================================================
# 8. FINAL DATA QUALITY CHECK
# ============================================================

print("Final dataset shape:")
print(df_clean.shape)

print("\nDuplicate rows:")
print(df_clean.duplicated().sum())

print("\nMissing values:")
display(df_clean.isna().sum().to_frame("Missing Values"))

print("\nData types:")
display(df_clean.dtypes.to_frame("Data Type"))

Final dataset shape:
(1200, 14)

Duplicate rows:
0

Missing values:


,Missing Values
OrderID,0
Date,0
CustomerID,0
Product,0
Quantity,0
UnitPrice,0
ShippingAddress,0
PaymentMethod,0
OrderStatus,0
TrackingNumber,0



Data types:


,Data Type
OrderID,object
Date,datetime64[ns]
CustomerID,object
Product,object
Quantity,int64
UnitPrice,float64
ShippingAddress,object
PaymentMethod,object
OrderStatus,object
TrackingNumber,object


## 9. Export Cleaned Dataset

The validated and standardized dataset is exported as an Excel file for downstream analytics and reporting.

In [32]:
# ============================================================
# 9. EXPORT CLEANED DATASET
# ============================================================

output_file = "Project_1_Cleaned_Dataset.xlsx"

df_clean.to_excel(
    output_file,
    index=False
)

print(f"Cleaned dataset exported successfully: {output_file}")

Cleaned dataset exported successfully: Project_1_Cleaned_Dataset.xlsx
